In [ ]:
from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_files = list(Path("/kaggle/input").rglob("train_images.npy"))
scripts = list(Path("/kaggle/input").rglob("task3_5fold_oof_colab.py"))

print("Data folder:", data_files[0].parent if data_files else "NOT FOUND")
print("Script:", scripts[0] if scripts else "NOT FOUND")

assert torch.cuda.is_available(), "Enable a GPU under Notebook Settings → Accelerator."
assert len(data_files) == 1, "Attach the dataset containing train_images.npy."
assert len(scripts) == 1, "Attach the dataset containing task3_5fold_oof_colab.py."

In [ ]:
import os
import runpy
from pathlib import Path

data_dir = next(Path("/kaggle/input").rglob("train_images.npy")).parent
script_path = next(Path("/kaggle/input").rglob("task3_5fold_oof_colab.py"))

os.environ["TASK3_DATA_DIR"] = str(data_dir)
os.environ["TASK3_OUTPUT_DIR"] = "/kaggle/working/task3_5fold_oof"

runpy.run_path(str(script_path), run_name="__main__")

Device: cuda. Five fresh ResNet-18 folds, up to 60 epochs each.
Fold 0: 36000 train, 9000 held out; resuming at epoch 1
Fold 0 | epoch 01/60 | train CE 2.19713 | heldout CE 2.05800
Fold 0 | epoch 02/60 | train CE 1.94115 | heldout CE 1.91597
Fold 0 | epoch 03/60 | train CE 1.86565 | heldout CE 2.24538
Fold 0 | epoch 04/60 | train CE 1.79955 | heldout CE 2.00849
Fold 0 | epoch 05/60 | train CE 1.74741 | heldout CE 1.83131
Fold 0 | epoch 06/60 | train CE 1.70404 | heldout CE 1.79767
Fold 0 | epoch 07/60 | train CE 1.67262 | heldout CE 1.74402
Fold 0 | epoch 08/60 | train CE 1.65196 | heldout CE 1.76230
Fold 0 | epoch 09/60 | train CE 1.62963 | heldout CE 1.83357
Fold 0 | epoch 10/60 | train CE 1.61042 | heldout CE 1.78468
Fold 0 | epoch 11/60 | train CE 1.59363 | heldout CE 1.67698
Fold 0 | epoch 12/60 | train CE 1.56611 | heldout CE 1.69753
Fold 0 | epoch 13/60 | train CE 1.54881 | heldout CE 1.75841
Fold 0 | epoch 14/60 | train CE 1.53649 | heldout CE 1.65856
Fold 0 | epoch 15/60 | tra

In [1]:

import os

print("Checking Kaggle input files:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename in {
            "train_images.npy",
            "test_images.npy",
            "labels.csv",
            "train_idx.txt",
            "val_idx.txt",
            "model.py",
            "t3_noise_scores_improved.csv",
        }:
            print(os.path.join(root, filename))


Checking Kaggle input files:

/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/labels.csv
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/model.py
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/t3_noise_scores_improved.csv
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/test_images.npy
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/train_idx.txt
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/train_images.npy
/kaggle/input/datasets/tvjpro/inter-iit-cv-ps/val_idx.txt


In [2]:

import os
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/tvjpro/inter-iit-cv-ps"

scores_path = os.path.join(DATA_DIR, "t3_noise_scores_improved.csv")
labels_path = os.path.join(DATA_DIR, "labels.csv")
train_idx_path = os.path.join(DATA_DIR, "train_idx.txt")
val_idx_path = os.path.join(DATA_DIR, "val_idx.txt")

scores_df = pd.read_csv(scores_path)
labels_df = pd.read_csv(labels_path)

train_idx = np.loadtxt(train_idx_path, dtype=int)
val_idx = np.loadtxt(val_idx_path, dtype=int)

print("Score columns:", scores_df.columns.tolist())
print("Score rows:", len(scores_df))
print("Unique score indices:", scores_df["index"].nunique())
print("Training indices:", len(train_idx))
print("Validation indices:", len(val_idx))

# Verify that every training index has a score
score_indices = set(scores_df["index"].astype(int))
print("\nAll training indices have scores:",
      set(train_idx).issubset(score_indices))

# Check score range and distribution
print("\nScore summary:")
print(scores_df["score"].describe())

# Show how many examples would remain under different thresholds
train_scores = scores_df.set_index("index").loc[train_idx, "score"].to_numpy()

for threshold in [0.3, 0.5, 0.7, 0.8]:
    kept = np.sum(train_scores <= threshold)
    print(
        f"Threshold {threshold:.1f}: keep {kept:,}/45,000 "
        f"({100 * kept / len(train_scores):.1f}%)"
    )


Score columns: ['index', 'score']
Score rows: 45000
Unique score indices: 45000
Training indices: 45000
Validation indices: 5000

All training indices have scores: True

Score summary:
count    45000.000000
mean         0.254827
std          0.291758
min          0.000048
25%          0.036380
50%          0.121792
75%          0.380678
max          0.999633
Name: score, dtype: float64
Threshold 0.3: keep 31,614/45,000 (70.3%)
Threshold 0.5: keep 36,201/45,000 (80.4%)
Threshold 0.7: keep 39,118/45,000 (86.9%)
Threshold 0.8: keep 40,491/45,000 (90.0%)


In [3]:

import numpy as np
import pandas as pd

THRESHOLD = 0.7

# Map each training index to its Task 3 noise score
score_map = scores_df.set_index("index")["score"]

# Align scores with the training split's index order
train_scores = score_map.loc[train_idx].to_numpy()

# Keep examples whose scores are at or below the threshold
keep_mask = train_scores <= THRESHOLD
filtered_train_idx = train_idx[keep_mask]
removed_train_idx = train_idx[~keep_mask]

print("Filtering threshold:", THRESHOLD)
print("Original training examples:", len(train_idx))
print("Retained examples:", len(filtered_train_idx))
print("Filtered-out examples:", len(removed_train_idx))
print("Retained percentage:",
      round(100 * len(filtered_train_idx) / len(train_idx), 2), "%")

# Use only noisy labels for training
noisy_label_map = labels_df.set_index("index")["noisy_label"]
filtered_noisy_labels = noisy_label_map.loc[filtered_train_idx].to_numpy()

print("\nRetained labels:", len(filtered_noisy_labels))
print("Label range:", filtered_noisy_labels.min(),
      "to", filtered_noisy_labels.max())
print("Class counts:")
print(pd.Series(filtered_noisy_labels).value_counts().sort_index())


Filtering threshold: 0.7
Original training examples: 45000
Retained examples: 39118
Filtered-out examples: 5882
Retained percentage: 86.93 %

Retained labels: 39118
Label range: 0 to 9
Class counts:
0    3893
1    4540
2    4365
3    3651
4    3053
5    4506
6    3410
7    4327
8    3749
9    3624
Name: count, dtype: int64


In [ ]:

import os
import sys
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# --------------------------------------------------
# Configuration
# --------------------------------------------------
SEED = 42
EPOCHS = 60
BATCH_SIZE = 128
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = "/kaggle/input/datasets/tvjpro/inter-iit-cv-ps"
OUTPUT_DIR = "/kaggle/working/task4_filtering"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# Load images and noisy labels only
# --------------------------------------------------
images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
labels_df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))

noisy_label_map = labels_df.set_index("index")["noisy_label"]

# filtered_train_idx and val_idx were created earlier
# from train_idx.txt and val_idx.txt.
train_targets = noisy_label_map.loc[filtered_train_idx].to_numpy(dtype=np.int64)
val_targets = noisy_label_map.loc[val_idx].to_numpy(dtype=np.int64)

# --------------------------------------------------
# Transformations
# --------------------------------------------------
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

class FilteredDataset(Dataset):
    def __init__(self, images, indices, targets, transform):
        self.images = images
        self.indices = np.asarray(indices, dtype=np.int64)
        self.targets = np.asarray(targets, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        image = self.transform(self.images[self.indices[i]])
        label = int(self.targets[i])
        return image, label

train_dataset = FilteredDataset(
    images, filtered_train_idx, train_targets, train_transform
)

val_dataset = FilteredDataset(
    images, val_idx, val_targets, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

# --------------------------------------------------
# Exact provided ResNet-18, trained from scratch
# --------------------------------------------------
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

from model import ResNet18

model_filter = ResNet18(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model_filter.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

best_val_acc = -1.0
best_epoch = 0
history = []
checkpoint_path = os.path.join(
    OUTPUT_DIR, "t4_filtering_best.pth"
)

# --------------------------------------------------
# Training
# --------------------------------------------------
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model_filter.train()
    total_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model_filter(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        train_correct += (logits.argmax(1) == y).sum().item()
        train_total += y.size(0)

    train_loss = total_loss / train_total
    train_acc = 100.0 * train_correct / train_total

    # Full validation set; use noisy labels only
    model_filter.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model_filter(x)
            loss = criterion(logits, y)

            val_loss_sum += loss.item() * y.size(0)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = 100.0 * val_correct / val_total
    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc_noisy": train_acc,
        "val_loss_noisy": val_loss,
        "val_acc_noisy": val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model_filter.state_dict(), checkpoint_path)

    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "t4_filtering_history.csv"),
        index=False
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc:.2f}% | "
        f"Noisy val acc: {val_acc:.2f}% | "
        f"Best: {best_val_acc:.2f}% (epoch {best_epoch}) | "
        f"{time.time() - start:.1f}s"
    )

print("\nTraining complete.")
print("Best noisy validation accuracy:", round(best_val_acc, 2), "%")
print("Best epoch:", best_epoch)
print("Best checkpoint:", checkpoint_path)


Device: cuda
Epoch 01/60 | Train loss: 2.2079 | Train acc: 23.07% | Noisy val acc: 17.54% | Best: 17.54% (epoch 1) | 31.8s
Epoch 02/60 | Train loss: 1.8262 | Train acc: 33.14% | Noisy val acc: 27.54% | Best: 27.54% (epoch 2) | 31.3s
Epoch 03/60 | Train loss: 1.7256 | Train acc: 37.33% | Noisy val acc: 28.24% | Best: 28.24% (epoch 3) | 32.1s
Epoch 04/60 | Train loss: 1.6397 | Train acc: 41.94% | Noisy val acc: 32.38% | Best: 32.38% (epoch 4) | 33.3s
Epoch 05/60 | Train loss: 1.5704 | Train acc: 45.08% | Noisy val acc: 41.48% | Best: 41.48% (epoch 5) | 34.7s
Epoch 06/60 | Train loss: 1.5109 | Train acc: 47.53% | Noisy val acc: 34.46% | Best: 41.48% (epoch 5) | 36.5s
Epoch 07/60 | Train loss: 1.4582 | Train acc: 49.97% | Noisy val acc: 32.90% | Best: 41.48% (epoch 5) | 36.2s
Epoch 08/60 | Train loss: 1.4219 | Train acc: 51.50% | Noisy val acc: 41.14% | Best: 41.48% (epoch 5) | 35.9s
Epoch 09/60 | Train loss: 1.3920 | Train acc: 53.14% | Noisy val acc: 44.44% | Best: 44.44% (epoch 9) | 36.

In [1]:

import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATA_DIR = "/kaggle/input/datasets/tvjpro/inter-iit-cv-ps"
OUT_DIR = "/kaggle/working/task4_coteaching"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
EPOCHS = 60
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MAX_FORGET_RATE = 0.20
RAMPUP_EPOCHS = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_images = np.load(f"{DATA_DIR}/train_images.npy")
test_images = np.load(f"{DATA_DIR}/test_images.npy")

labels_df = pd.read_csv(f"{DATA_DIR}/labels.csv")
train_idx = np.loadtxt(f"{DATA_DIR}/train_idx.txt", dtype=np.int64)
val_idx = np.loadtxt(f"{DATA_DIR}/val_idx.txt", dtype=np.int64)

# Task 4 uses noisy labels only.
noisy_labels = labels_df.set_index("index")["noisy_label"]
train_labels = noisy_labels.loc[train_idx].to_numpy(dtype=np.int64)
val_labels = noisy_labels.loc[val_idx].to_numpy(dtype=np.int64)

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

class CIFARArrayDataset(Dataset):
    def __init__(self, images, indices, labels=None, transform=None):
        self.images = images
        self.indices = np.asarray(indices)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        image = self.images[idx]
        if self.transform:
            image = self.transform(image)
        if self.labels is None:
            return image, idx
        return image, int(self.labels[i])

train_ds = CIFARArrayDataset(
    train_images, train_idx, train_labels, train_transform
)
val_ds = CIFARArrayDataset(
    train_images, val_idx, val_labels, eval_transform
)
test_ds = CIFARArrayDataset(
    test_images, np.arange(len(test_images)), None, eval_transform
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

sys.path.insert(0, DATA_DIR)
from model import ResNet18

# Two independent networks, both trained from scratch.
net1 = ResNet18(num_classes=10).to(device)
net2 = ResNet18(num_classes=10).to(device)

optimizer1 = torch.optim.SGD(
    net1.parameters(), lr=LR, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)
optimizer2 = torch.optim.SGD(
    net2.parameters(), lr=LR, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer1, T_max=EPOCHS
)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Output directory:", OUT_DIR)


Device: cuda
Train: 45000
Validation: 5000
Test: 10000
Output directory: /kaggle/working/task4_coteaching


In [2]:

def evaluate(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)

            loss_sum += F.cross_entropy(
                logits, labels, reduction="sum"
            ).item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return loss_sum / total, correct / total


history = []
best_val_acc = -1.0
best_epoch = -1

for epoch in range(1, EPOCHS + 1):
    net1.train()
    net2.train()

    # Ramp up the fraction discarded during the first 10 epochs.
    forget_rate = MAX_FORGET_RATE * min(
        1.0, (epoch - 1) / max(1, RAMPUP_EPOCHS - 1)
    )

    loss_sum1 = 0.0
    loss_sum2 = 0.0
    num_seen = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        optimizer2.zero_grad(set_to_none=True)

        logits1 = net1(images)
        logits2 = net2(images)

        # Rank examples by each network's own per-example loss.
        with torch.no_grad():
            per_example_loss1 = F.cross_entropy(
                logits1, labels, reduction="none"
            )
            per_example_loss2 = F.cross_entropy(
                logits2, labels, reduction="none"
            )

            remember = max(
                1, int((1.0 - forget_rate) * labels.size(0))
            )
            selected1 = torch.argsort(per_example_loss1)[:remember]
            selected2 = torch.argsort(per_example_loss2)[:remember]

        # Cross-update: each network learns from the other's selection.
        loss1 = F.cross_entropy(logits1[selected2], labels[selected2])
        loss2 = F.cross_entropy(logits2[selected1], labels[selected1])

        loss1.backward()
        loss2.backward()
        optimizer1.step()
        optimizer2.step()

        n = labels.size(0)
        loss_sum1 += loss1.item() * n
        loss_sum2 += loss2.item() * n
        num_seen += n

    scheduler1.step()
    scheduler2.step()

    val_loss1, val_acc1 = evaluate(net1, val_loader)
    val_loss2, val_acc2 = evaluate(net2, val_loader)

    if val_acc1 >= val_acc2:
        selected_model = net1
        selected_name = "net1"
        selected_acc = val_acc1
    else:
        selected_model = net2
        selected_name = "net2"
        selected_acc = val_acc2

    row = {
        "epoch": epoch,
        "forget_rate": forget_rate,
        "train_loss_net1": loss_sum1 / num_seen,
        "train_loss_net2": loss_sum2 / num_seen,
        "val_loss_net1": val_loss1,
        "val_acc_net1": val_acc1,
        "val_loss_net2": val_loss2,
        "val_acc_net2": val_acc2,
        "selected_network": selected_name,
        "selected_val_acc": selected_acc,
    }
    history.append(row)

    # Save progress every epoch, even if it is not the best epoch.
    checkpoint = {
        "epoch": epoch,
        "net1_state_dict": net1.state_dict(),
        "net2_state_dict": net2.state_dict(),
        "selected_state_dict": selected_model.state_dict(),
        "selected_network": selected_name,
        "selected_val_acc": selected_acc,
        "forget_rate": forget_rate,
    }

    torch.save(checkpoint, f"{OUT_DIR}/t4_coteaching_latest.pth")
    pd.DataFrame(history).to_csv(
        f"{OUT_DIR}/t4_coteaching_history.csv", index=False
    )

    if selected_acc > best_val_acc:
        best_val_acc = selected_acc
        best_epoch = epoch
        torch.save(checkpoint, f"{OUT_DIR}/t4_coteaching_best.pth")

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"forget={forget_rate:.3f} | "
        f"Net1 val={val_acc1*100:.2f}% | "
        f"Net2 val={val_acc2*100:.2f}% | "
        f"Best={best_val_acc*100:.2f}% (epoch {best_epoch})"
    )

print(f"Best noisy validation accuracy: {best_val_acc*100:.2f}%")
print("Best epoch:", best_epoch)


Epoch 01/60 | forget=0.000 | Net1 val=27.44% | Net2 val=27.30% | Best=27.44% (epoch 1)
Epoch 02/60 | forget=0.022 | Net1 val=25.76% | Net2 val=31.76% | Best=31.76% (epoch 2)
Epoch 03/60 | forget=0.044 | Net1 val=29.60% | Net2 val=29.24% | Best=31.76% (epoch 2)
Epoch 04/60 | forget=0.067 | Net1 val=25.28% | Net2 val=36.92% | Best=36.92% (epoch 4)
Epoch 05/60 | forget=0.089 | Net1 val=21.28% | Net2 val=26.42% | Best=36.92% (epoch 4)
Epoch 06/60 | forget=0.111 | Net1 val=42.24% | Net2 val=39.84% | Best=42.24% (epoch 6)
Epoch 07/60 | forget=0.133 | Net1 val=35.68% | Net2 val=43.18% | Best=43.18% (epoch 7)
Epoch 08/60 | forget=0.156 | Net1 val=41.74% | Net2 val=40.20% | Best=43.18% (epoch 7)
Epoch 09/60 | forget=0.178 | Net1 val=46.52% | Net2 val=44.32% | Best=46.52% (epoch 9)
Epoch 10/60 | forget=0.200 | Net1 val=44.60% | Net2 val=46.46% | Best=46.52% (epoch 9)
Epoch 11/60 | forget=0.200 | Net1 val=47.30% | Net2 val=46.36% | Best=47.30% (epoch 11)
Epoch 12/60 | forget=0.200 | Net1 val=44.9

In [3]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

def find_file(filename):
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} under /kaggle/working or /kaggle/input"
    )

checkpoint_path = find_file("t4_coteaching_best.pth")
model_path = find_file("model.py")

spec = importlib.util.spec_from_file_location("submission_model", model_path)
submission_model = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission_model)
ResNet18 = submission_model.ResNet18

if "test_images" not in globals():
    test_images = np.load(find_file("test_images.npy"), mmap_mode="r")

assert test_images.shape == (10_000, 32, 32, 3), test_images.shape

class TestImages(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = np.asarray(self.images[i])
        if image.dtype != np.uint8:
            image = np.clip(image, 0, 255).astype(np.uint8)
        return torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Selected network:", checkpoint.get("selected_network"))

model = ResNet18(num_classes=10).to(device)
model.load_state_dict(checkpoint["selected_state_dict"])
model.eval()

loader = DataLoader(TestImages(test_images), batch_size=256, shuffle=False)
predictions = []

with torch.inference_mode():
    for batch in loader:
        predictions.extend(model(batch.to(device)).argmax(dim=1).cpu().tolist())

assert len(predictions) == 10_000
assert all(0 <= p <= 9 for p in predictions)

output_path = Path("/kaggle/working/t4_coteaching.csv")
pd.DataFrame({
    "id": np.arange(len(predictions)),
    "pred": predictions
}).to_csv(output_path, index=False)

print("Saved:", output_path)
display(FileLink(str(output_path)))

Checkpoint epoch: 40
Selected network: net1
Saved: /kaggle/working/t4_coteaching.csv


/kaggle/working/t4_coteaching.csv